Exercise 1: Customer Review to Support Ticket Pipeline.

In [1]:
!pip install -q langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.1 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq


In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


In [4]:
llm = ChatGroq(
    model ="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)

In [5]:
response = llm.invoke("helloooooo")

In [6]:
print(response)

content='Hello! 👋 How can I help you today?' additional_kwargs={'reasoning_content': 'The user says "helloooooo". This is a greeting. We should respond politely, maybe ask how we can help. The user hasn\'t specified a question. So respond with friendly greeting.'} response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 74, 'total_tokens': 133, 'completion_time': 0.066795235, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003611776, 'prompt_tokens_details': None, 'queue_time': 0.166985418, 'total_time': 0.070407011}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a166de-5111-7da2-932b-a4a9b72419bc-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 74, 'output_tokens': 59, 'total_tokens': 133, 'output_token_details': {'reasoning': 39}}


In [7]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

In [9]:
class ComplaintData(BaseModel):                                  #here we have created our own pydantic model called ComplaintData.
  complaint: str = Field(description="Main Complaint of the customer")
  product_or_feature: str = Field(description="Product or feature mentioned")
  sentiment: str = Field(description="Customer sentiment")

In [10]:
complaint_parser = PydanticOutputParser(pydantic_object=ComplaintData)

In [12]:
print(complaint_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"complaint": {"description": "Main Complaint of the customer", "title": "Complaint", "type": "string"}, "product_or_feature": {"description": "Product or feature mentioned", "title": "Product Or Feature", "type": "string"}, "sentiment": {"description": "Customer sentiment", "title": "Sentiment", "type": "string"}}, "required": ["complaint", "product_or_feature", "sentiment"]}
```


In [13]:
#Creating PromptTemplate
complaint_template = PromptTemplate.from_template(
    """
    Extract the complaint infromation from the following customer review.
    Customer_Review:{review_text}
    {PYDANTIC_FORMATTING}
    """
).partial(PYDANTIC_FORMATTING=complaint_parser.get_format_instructions())

In [14]:
complaint_input= complaint_template.invoke({
    "review_text":"I bought this laptop last week and the battery drains within two hours.I am very disappointed wuth the battery performance"
})

In [17]:
print(complaint_input)

text='\n    Extract the complaint infromation from the following customer review.\n    Customer_Review:I bought this laptop last week and the battery drains within two hours.I am very disappointed wuth the battery performance\n    The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"complaint": {"description": "Main Complaint of the customer", "title": "Complaint", "type": "string"}, "product_or_feature": {"description": "Product or feature mentioned", "title": "Product Or Feature", "type": "string"}, "sentiment": {"description": "Customer sentiment", "title": "Sentiment"

In [18]:
complaint_response = llm.invoke(complaint_input)

In [19]:
print(complaint_response.content)

{"complaint":"The battery drains within two hours","product_or_feature":"battery performance","sentiment":"negative"}


In [20]:
complaint_data = complaint_parser.invoke(complaint_response)

In [21]:
print(complaint_data)

complaint='The battery drains within two hours' product_or_feature='battery performance' sentiment='negative'


In [23]:
print(complaint_data.complaint)
print(complaint_data.product_or_feature)
print(complaint_data.sentiment)

The battery drains within two hours
battery performance
negative


Step 2 — Generate the Support Ticket

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [26]:
ticket_template = ChatPromptTemplate.from_messages([
    ("system", "You are a customer support assistent. Create a clear and professional support ticket."),
    ("user", """
    Complaint:{complaint}
    Product or Feature: {product_or_feature}
    Sentiment: {sentiment}
    Generate a short support ticket suammary.ChatPromptTemplate
    """)
])

In [28]:
ticket_parser = StrOutputParser()

In [29]:
ticket_input = ticket_template.invoke({
    "complaint": complaint_data.complaint,
    "product_or_feature": complaint_data.product_or_feature,
    "sentiment": complaint_data.sentiment
})

In [30]:
print(ticket_input)

messages=[SystemMessage(content='You are a customer support assistent. Create a clear and professional support ticket.', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n    Complaint:The battery drains within two hours\n    Product or Feature: battery performance\n    Sentiment: negative\n    Generate a short support ticket suammary.ChatPromptTemplate\n    ', additional_kwargs={}, response_metadata={})]


In [31]:
ticket_response = llm.invoke(ticket_input)


In [32]:
final_ticket = ticket_parser.invoke(ticket_response)

In [33]:
print(final_ticket)

**Support Ticket Summary**

- **Ticket ID:** 2026‑10‑23‑001  
- **Subject:** Battery drains within 2 hours  
- **Product/Feature:** Battery performance  
- **Customer Sentiment:** Negative  
- **Issue Description:** User reports rapid battery depletion, with the device losing charge within two hours of normal use.  
- **Requested Action:** Investigate root cause, provide troubleshooting steps, and recommend potential replacement or repair options.


## Exercise 2 — Research Paper Abstract to Layperson Summary

In [34]:
class ResearchData(BaseModel):
  research_problem: str = Field(description="Main Research Problem")
  method: str = Field(description="Method used in the research")
  key_result: str = Field(description="Main result of the research")

In [35]:
research_parser = PydanticOutputParser(
    pydantic_object=ResearchData
)

In [36]:
print(research_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"research_problem": {"description": "Main Research Problem", "title": "Research Problem", "type": "string"}, "method": {"description": "Method used in the research", "title": "Method", "type": "string"}, "key_result": {"description": "Main result of the research", "title": "Key Result", "type": "string"}}, "required": ["research_problem", "method", "key_result"]}
```


Step 1 — Create the Research Prompt

In [37]:
research_template = PromptTemplate.from_template(
    """
    Extract the important research information from the following abstract.

    Research Abstract:
    {abstract_text}

    {PYDANTIC_FORMATTING}
    """
).partial(
    PYDANTIC_FORMATTING=research_parser.get_format_instructions()
)

In [38]:
research_input = research_template.invoke({
    "abstract_text": """
    This study investigates the problem of detecting fraudulent credit card transactions.
    A Random Forest machine learning model was trained using historical transaction data.
    The proposed model achieved 94% accuracy in identifying fraudulent transactions.
    """
})

In [39]:
print(research_input)


text='\n    Extract the important research information from the following abstract.\n\n    Research Abstract:\n    \n    This study investigates the problem of detecting fraudulent credit card transactions.\n    A Random Forest machine learning model was trained using historical transaction data.\n    The proposed model achieved 94% accuracy in identifying fraudulent transactions.\n    \n\n    The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"research_problem": {"description": "Main Research Problem", "title": "Research Problem", "type": "string"}, "method": {"descript

In [40]:
research_response = llm.invoke(research_input)

In [41]:
print(research_response.content)

{"research_problem":"detecting fraudulent credit card transactions","method":"Random Forest machine learning model trained on historical transaction data","key_result":"94% accuracy in identifying fraudulent transactions"}


In [42]:
research_data = research_parser.invoke(research_response)

In [43]:
print("Research Problem:", research_data.research_problem)
print("Method:", research_data.method)
print("Key Result:", research_data.key_result)

Research Problem: detecting fraudulent credit card transactions
Method: Random Forest machine learning model trained on historical transaction data
Key Result: 94% accuracy in identifying fraudulent transactions


Step 2 — Generate Layperson Summary

In [44]:
summary_template = ChatPromptTemplate.from_messages([
    ("system", "You explain research in simple language that a non-technical person can understand."),

    ("user", """
    Research Problem: {research_problem}
    Method: {method}
    Key Result: {key_result}

    Write a short and simple layperson summary of this research.
    """)
])

In [45]:
summary_parser = StrOutputParser()

In [46]:
summary_input = summary_template.invoke({
    "research_problem": research_data.research_problem,
    "method": research_data.method,
    "key_result": research_data.key_result
})

In [47]:
print(summary_input)

messages=[SystemMessage(content='You explain research in simple language that a non-technical person can understand.', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n    Research Problem: detecting fraudulent credit card transactions\n    Method: Random Forest machine learning model trained on historical transaction data\n    Key Result: 94% accuracy in identifying fraudulent transactions\n\n    Write a short and simple layperson summary of this research.\n    ', additional_kwargs={}, response_metadata={})]


In [48]:
summary_response = llm.invoke(summary_input)

In [49]:
final_summary = summary_parser.invoke(summary_response)

In [50]:
print(final_summary)

**What we did**  
We wanted to spot credit‑card fraud—transactions that are fake or stolen—so banks can stop them before money is lost.

**How we did it**  
We used a computer “brain” called a Random Forest. It learns from thousands of past transactions, looking at patterns like how much was spent, where, and when. After training, it can guess whether a new transaction is likely fraud or not.

**What we found**  
When we tested it, the model correctly identified 94 % of fraudulent transactions. That means it only missed about 6 % of the bad ones, which is a strong performance for protecting customers and banks.
